In [8]:
import pandas as pd
import numpy as np
import os
import re
import joblib
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

# Buscador inteligente de la carpeta 'data'
if os.path.exists('../data/dataset_ia_final.csv'):
    PATH_DATA = '../data'
elif os.path.exists('data/dataset_ia_final.csv'):
    PATH_DATA = 'data'
elif os.path.exists('dataset_ia_final.csv'):
    PATH_DATA = '.'
else:
    raise FileNotFoundError("❌ No encuentro el archivo 'dataset_ia_final.csv' por ninguna parte. Revisa los nombres.")

archivo = os.path.join(PATH_DATA, 'dataset_ia_final.csv')
print(f"✅ Carpeta de datos localizada en: {PATH_DATA}")
print(f"✅ Archivo a leer: {archivo}")

✅ Carpeta de datos localizada en: data
✅ Archivo a leer: data\dataset_ia_final.csv


In [9]:
print("⏳ Cargando datos en memoria...")
df = pd.read_csv(archivo, low_memory=False)

# 1. Función para limpiar la columna de baños (ej: "1 bath" -> 1.0)
def extraer_banos(texto):
    if pd.isna(texto): return 1.0
    numeros = re.findall(r"[-+]?\d*\.\d+|\d+", str(texto))
    return float(numeros[0]) if numeros else 1.0

df['bathrooms_num'] = df['bathrooms_text'].apply(extraer_banos)

# 2. Tratamiento de nulos de seguridad
# Si en algún barrio falló la renta, le ponemos la media de Sevilla
df['renta_media'] = df['renta_media'].fillna(df['renta_media'].median())

# Si algún piso no tiene habitaciones, asumimos 1
df['bedrooms'] = df['bedrooms'].fillna(1.0)

# El dataset que me has pasado parece tener el sentimiento como última columna. 
# Si por algún motivo no se llama 'score_sentimiento', creamos una neutra de respaldo.
if 'score_sentimiento' not in df.columns:
    print("⚠️ No detecto la cabecera 'score_sentimiento', usando valor neutro por defecto.")
    df['score_sentimiento'] = 0.5
else:
    df['score_sentimiento'] = df['score_sentimiento'].fillna(0.5)

print(f"✅ Datos limpios y listos. Total de alojamientos: {len(df)}")

⏳ Cargando datos en memoria...
✅ Datos limpios y listos. Total de alojamientos: 8215


In [11]:
print("🧠 Iniciando entrenamiento del modelo KNN (Versión Segura)...")

# --- 1. ESCUDOS ANTI-ERRORES ---

# Escudo A: Limpiar el precio (A veces Airbnb manda el precio con $ o comas)
if df['price'].dtype == 'object':
    print("🧹 Limpiando columna de precios (quitando símbolos)...")
    # Quitamos cualquier cosa que no sea un número o un punto decimal
    df['price'] = df['price'].astype(str).str.replace(r'[^\d.]', '', regex=True)
    # Convertimos strings vacíos a NaN para que no fallen al pasar a float
    df['price'] = pd.to_numeric(df['price'], errors='coerce')

# Escudo B: Salvar la Renta Media
if df['renta_media'].isna().all():
    print("⚠️ ALERTA: Tu columna 'renta_media' está completamente vacía. Aplicando 15000€ por defecto.")
    df['renta_media'] = 15000.0
else:
    df['renta_media'] = df['renta_media'].fillna(df['renta_media'].median())

# --- 2. PREPARACIÓN DE DATOS ---

features = ['latitude', 'longitude', 'accommodates', 'bedrooms', 'bathrooms_num', 'renta_media', 'score_sentimiento']
target = 'price'

# Ahora sí, borramos nulos sabiendo que no vamos a perder todo
df_knn = df.dropna(subset=[target] + features).copy()

print(f"📊 Filas que han sobrevivido a la limpieza: {len(df_knn)}")

# --- 3. ENTRENAMIENTO ---

if len(df_knn) == 0:
    print("❌ ERROR CRÍTICO: Nos seguimos quedando con 0 datos.")
    print("Diagnóstico de nulos por columna:")
    print(df[features + [target]].isna().sum())
else:
    X = df_knn[features]
    y = df_knn[target]

    # ESCALADO
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # ENTRENAMIENTO
    knn_model = KNeighborsRegressor(n_neighbors=5, weights='distance')
    knn_model.fit(X_scaled, y)
    print(f"✅ ¡ÉXITO! Modelo entrenado con {len(df_knn)} alojamientos.")

    # GUARDADO
    ruta_modelo = os.path.join(PATH_DATA, 'knn_model.pkl')
    ruta_scaler = os.path.join(PATH_DATA, 'scaler.pkl')

    joblib.dump(knn_model, ruta_modelo)
    joblib.dump(scaler, ruta_scaler)

    print(f"💾 Cerebro de la IA exportado correctamente a:")
    print(f" -> {ruta_modelo}")

🧠 Iniciando entrenamiento del modelo KNN (Versión Segura)...
⚠️ ALERTA: Tu columna 'renta_media' está completamente vacía. Aplicando 15000€ por defecto.
📊 Filas que han sobrevivido a la limpieza: 7581
✅ ¡ÉXITO! Modelo entrenado con 7581 alojamientos.
💾 Cerebro de la IA exportado correctamente a:
 -> data\knn_model.pkl


In [ ]:
import streamlit as st
import pandas as pd
import joblib
import os
import numpy as np

st.set_page_config(page_title="Tasador Airbnb Sevilla", page_icon="🏠")

# Buscador inteligente de archivos para Streamlit
if os.path.exists('data/knn_model.pkl'):
    PATH_DATA = 'data'
elif os.path.exists('../data/knn_model.pkl'):
    PATH_DATA = '../data'
else:
    PATH_DATA = '.'

try:
    knn_model = joblib.load(os.path.join(PATH_DATA, 'knn_model.pkl'))
    scaler = joblib.load(os.path.join(PATH_DATA, 'scaler.pkl'))
except FileNotFoundError:
    st.error("❌ Faltan los archivos del modelo. Ejecuta primero tu notebook de KNN.")
    st.stop()

st.title("🏠 Tasador Inteligente Airbnb - Sevilla")
st.markdown("Esta herramienta utiliza Inteligencia Artificial (**K-Nearest Neighbors**) para analizar el mercado en tiempo real y sugerir el precio óptimo por noche.")

# Interfaz gráfica
col1, col2 = st.columns(2)

with col1:
    st.subheader("Características del Inmueble")
    accommodates = st.number_input("Capacidad (Personas)", 1, 16, 2)
    bedrooms = st.number_input("Habitaciones", 1, 10, 1)
    bathrooms = st.number_input("Baños", 0.5, 10.0, 1.0, step=0.5)

with col2:
    st.subheader("Contexto de la Zona")
    renta_media = st.number_input("Renta media del barrio (€)", 5000, 40000, 15000)
    score_sentimiento = st.slider("Calidad de reseñas esperada (0-1)", 0.0, 1.0, 0.8)
    lat = st.number_input("Latitud (Ej: 37.388)", value=37.388, format="%.5f")
    lon = st.number_input("Longitud (Ej: -5.999)", value=-5.999, format="%.5f")

# Botón mágico
if st.button("💰 Calcular Precio Óptimo", type="primary"):
    # Orden estricto: lat, lon, acc, bed, bath, renta, score
    input_data = np.array([[lat, lon, accommodates, bedrooms, bathrooms, renta_media, score_sentimiento]])
    
    input_scaled = scaler.transform(input_data)
    prediccion = knn_model.predict(input_scaled)[0]
    margen = prediccion * 0.15 # Asumimos 15% de elasticidad de precio
    
    st.success(f"### 💶 Precio Sugerido: {prediccion:.2f} € / noche")
    st.info(f"**Rango recomendado:** {prediccion-margen:.2f} € - {prediccion+margen:.2f} €")

2026-04-08 20:55:34.012 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-08 20:55:34.023 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-08 20:55:34.383 
  command:

    streamlit run c:\Users\delga\AppData\Local\Programs\Python\Python313\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-08 20:55:34.385 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-08 20:55:34.387 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-08 20:55:34.388 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-08 20:55:34.388 Thread 'MainThread': missing ScriptRunContext! This wa